In [1]:
!pip install streamlit scikit-learn pandas numpy nltk PyPDF2 python-docx

In [2]:
!pip install streamlit scikit-learn numpy nltk PyPDF2 python-docx
# Note: pandas is removed from the list

In [3]:
!pip install streamlit==1.32.0 scikit-learn==1.4.0 pandas==2.2.2 numpy==1.26.4 nltk PyPDF2 python-docx

In [4]:
import streamlit as st
import pandas as pd
import numpy as np
import nltk
import re
import io
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from PyPDF2 import PdfReader
from docx import Document

# Download NLTK data (run once)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('taggers/averaged_perceptron_tagger')
except LookupError:
    nltk.download('averaged_perceptron_tagger')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# --- Configuration ---
st.set_page_config(page_title="Smart Resume Screening", page_icon="🎯", layout="wide")

# --- Helper Functions ---

def preprocess_text(text):
    """Clean text: lower, remove non-alpha, tokenize, remove stopwords, lemmatize."""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()

    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    filtered_tokens = []
    for word in tokens:
        if word not in stop_words and len(word) > 2:
            filtered_tokens.append(lemmatizer.lemmatize(word))

    return " ".join(filtered_tokens)

def extract_text_from_pdf(file_bytes):
    """Extract text from PDF file."""
    reader = PdfReader(io.BytesIO(file_bytes))
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text

def extract_text_from_docx(file_bytes):
    """Extract text from DOCX file."""
    doc = Document(io.BytesIO(file_bytes))
    text = "\n".join([para.text for para in doc.paragraphs])
    return text

def calculate_similarity(resume_text, job_desc_text):
    """Calculate cosine similarity between resume and job description."""
    # Preprocess
    clean_resume = preprocess_text(resume_text)
    clean_job = preprocess_text(job_desc_text)

    if not clean_resume or not clean_job:
        return 0.0, [], []

    # Vectorize
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([clean_resume, clean_job])

    # Calculate Similarity
    similarity_score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]

    # Get Top Keywords (Intersection)
    resume_words = set(clean_resume.split())
    job_words = set(clean_job.split())
    common_keywords = list(resume_words.intersection(job_words))

    return similarity_score, sorted(common_keywords, key=len, reverse=True)[:10], list(resume_words)[:10]

# --- UI Layout ---

st.title("🎯 Smart Resume Screening System")
st.markdown("""
This system uses **Natural Language Processing (NLP)** and **Cosine Similarity** to rank resumes against job descriptions.
""")

# Sidebar
with st.sidebar:
    st.header("⚙️ Settings")
    st.info("Upload a resume (PDF/DOCX/TXT) and paste the Job Description.")
    st.markdown("---")
    st.markdown("**Tech Stack:**")
    st.markdown("- Python 3")
    st.markdown("- Scikit-learn (TF-IDF)")
    st.markdown("- NLTK (Preprocessing)")
    st.markdown("- Streamlit (UI)")

# Main Input Area
col1, col2 = st.columns(2)

with col1:
    st.subheader("📄 Resume Upload")
    uploaded_file = st.file_uploader("Choose a file", type=['pdf', 'docx', 'txt'])

    resume_text = ""
    if uploaded_file is not None:
        try:
            if uploaded_file.name.endswith('.pdf'):
                resume_text = extract_text_from_pdf(uploaded_file.read())
            elif uploaded_file.name.endswith('.docx'):
                resume_text = extract_text_from_docx(uploaded_file.read())
            else:
                resume_text = uploaded_file.read().decode("utf-8")

            st.success("File loaded successfully!")
            with st.expander("Preview Extracted Text"):
                st.text_area("Extracted Content", value=resume_text[:500] + "...", height=150)
        except Exception as e:
            st.error(f"Error reading file: {e}")

with col2:
    st.subheader("💼 Job Description")
    job_desc = st.text_area("Paste the Job Description here:", height=250,
                            placeholder="e.g. We are looking for a Python Developer with experience in Machine Learning, Pandas, and Scikit-learn...")

# Analysis Button
if st.button("🔍 Analyze Match", type="primary"):
    if not uploaded_file:
        st.warning("Please upload a resume file.")
    elif not job_desc.strip():
        st.warning("Please enter a Job Description.")
    else:
        with st.spinner("Processing NLP and calculating similarity..."):
            score, common_kw, resume_kw = calculate_similarity(resume_text, job_desc)

            # Determine Rating
            if score >= 0.6:
                rating = "✅ Excellent Match"
                color = "#22d3ae" # Green/Teal
            elif score >= 0.4:
                rating = "⚠️ Good Potential"
                color = "#fbbf24" # Yellow
            else:
                rating = "❌ Low Match"
                color = "#f87171" # Red

            # Display Results
            st.markdown("---")
            st.markdown(f"### 📊 Result: {rating}")
            st.progress(score)
            st.metric(label="Similarity Score", value=f"{score:.2%}", delta_color="normal")

            col_res, col_job = st.columns(2)

            with col_res:
                st.subheader("🔑 Key Skills Found")
                if common_kw:
                    st.write(", ".join(common_kw))
                else:
                    st.write("No direct keyword overlap found.")

                st.markdown("**Resume Keywords:**")
                st.write(", ".join(resume_kw))

            with col_job:
                st.subheader("📝 Match Details")
                st.write(f"The system identified **{len(common_kw)}** shared keywords between the resume and the job description.")
                st.caption("Analysis based on TF-IDF vectorization and Cosine Similarity.")

# Footer
st.markdown("---")
st.markdown("<div style='text-align: center; color: gray;'>Built with Streamlit & Scikit-learn | Smart Resume Screening Project</div>", unsafe_allow_html=True)

2026-03-14 15:51:10.519 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]


DeltaGenerator()